# Final NEDI x3 and x4 Evaluation — AWS Multi-CPU

Runs both remaining scales in one pass. NEDI only doubles, so the higher scales cascade native x2 passes:

- **x3** — `LR_x3` → one native NEDI x2 → one bicubic resize to the exact HR size.
- **x4** — `LR_x4` → native NEDI x2 → native NEDI x2 → resize only if the prepared pair is not an exact 4x match.

Bicubic is rerun on the same AWS CPU at both scales so the latency comparison is fair. Results are checkpointed after every image and the run is resumable. A Set5 pilot gate must pass before the full run starts.

**Expected budget:** roughly 2 h wall for x3 and 6 h for x4 on 4 workers, dominated by Urban100 x4.

**Expected outcome:** NEDI should land about 0.1–0.5 dB *below* bicubic in PSNR-Y. That is the documented behaviour of NEDI under bicubic-downsampled benchmark pairs, not a failure — see `docs/notes/nedi_vs_bicubic_findings.md`.

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False
    print('AWS/local Jupyter environment detected.')

AWS/local Jupyter environment detected.


In [2]:
import os
import subprocess
from pathlib import Path

# Prevent each worker from secretly creating more numerical-library threads.
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'

if IN_COLAB:
    REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
    if REPO_ROOT.exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    # Jupyter commonly starts in notebooks/, rather than the repository root.
    start_dir = Path.cwd().resolve()
    REPO_ROOT = next((directory for directory in (start_dir, *start_dir.parents)
                      if (directory / 'app').is_dir()), None)
    if REPO_ROOT is None:
        raise RuntimeError(
            f'Could not find the repository above {start_dir}. '
            'Open this notebook from the repository or set its working directory to it.'
        )

os.chdir(REPO_ROOT)
print(f'Repository ready: {REPO_ROOT}')

Repository ready: /home/ubuntu/Code/SuperResolution-Comparative-Analysis


In [3]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.evaluation.data_validation import validate_prepared_dataset
from app.evaluation.bicubic import BicubicEvaluationConfig, evaluate_bicubic_image
from app.evaluation.experiment import write_results_csv
from app.evaluation.images import pair_image_paths
from app.evaluation.nedi import NEDIEvaluationConfig, evaluate_nedi_image
from app.evaluation.reporting import summarize_results
from app.traditional.nedi import NEDI_NATIVE_PASSES, NEDI_SCALE_STRATEGIES

print('Bicubic and NEDI x3/x4 evaluators imported successfully.')
print('Strategies:', NEDI_SCALE_STRATEGIES)

Bicubic and NEDI x3/x4 evaluators imported successfully.
Strategies: {2: 'native_x2', 3: 'native_x2_then_bicubic_to_target', 4: 'native_x2_twice_then_bicubic_to_target'}


In [4]:
from datetime import UTC, datetime
from concurrent.futures import ProcessPoolExecutor, as_completed
import csv

default_data_root = '/content/drive/MyDrive/FYP_SR_Data' if IN_COLAB else '/mnt/fyp-data/FYP_SR_Data'
DATA_ROOT = Path(os.environ.get('FYP_SR_DATA_ROOT', default_data_root))
WORKERS = int(os.environ.get('FYP_NEDI_WORKERS', '4'))
COMPUTE_INSTANCE = os.environ.get('FYP_COMPUTE_INSTANCE', 'm7i.2xlarge')
# Leave as None for a new run. To resume after an interruption, replace
# None with the timestamped folder name printed by the earlier run.
RESUME_RUN_ID = None
RUN_ID = RESUME_RUN_ID or datetime.now(UTC).strftime('%Y%m%d_%H%M%S_utc')
RUN_ROOT = DATA_ROOT / 'results' / 'final_nedi' / 'x3_x4_full' / RUN_ID
METRICS_ROOT = RUN_ROOT / 'metrics'
DATASETS = ('Set5', 'Set14', 'BSD100', 'Urban100')
SCALES = (3, 4)
WARMUP_RUNS = 3
TIMED_RUNS = 10
WINDOW_SIZE = 8
EDGE_THRESHOLD = 8.0

if WORKERS < 1:
    raise ValueError('FYP_NEDI_WORKERS must be at least 1.')
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset root not found: {DATA_ROOT}')

print(f'Full NEDI x3/x4 run: {RUN_ROOT}')
print(f'Compute: {COMPUTE_INSTANCE}; workers: {WORKERS}')
print(f'Protocol: {WARMUP_RUNS} warm-ups and {TIMED_RUNS} timed CPU runs per image.')
print('NEDI parameters frozen to match the completed x2 run: '
      f'window_size={WINDOW_SIZE}, edge_threshold={EDGE_THRESHOLD}')

Full NEDI x3/x4 run: /mnt/fyp-data/FYP_SR_Data/results/final_nedi/x3_x4_full/20260831_171327_utc
Compute: m7i.2xlarge; workers: 4
Protocol: 3 warm-ups and 10 timed CPU runs per image.
NEDI parameters frozen to match the completed x2 run: window_size=8, edge_threshold=8.0


In [5]:
validations = {}
for scale in SCALES:
    for dataset in DATASETS:
        validation = validate_prepared_dataset(dataset, scale, DATA_ROOT)
        validations[(dataset, scale)] = validation
        print(f'VALID: {dataset} x{scale} has {validation.image_count} complete HR/LR pairs.')

expected_total = sum(validation.image_count for validation in validations.values())
if expected_total != 438:
    raise RuntimeError(f'Expected 438 image evaluations (219 x 2 scales); found {expected_total}.')
print(f'\nAll x3 and x4 datasets passed validation: {expected_total} image evaluations queued.')

VALID: Set5 x3 has 5 complete HR/LR pairs.
VALID: Set14 x3 has 14 complete HR/LR pairs.
VALID: BSD100 x3 has 100 complete HR/LR pairs.
VALID: Urban100 x3 has 100 complete HR/LR pairs.
VALID: Set5 x4 has 5 complete HR/LR pairs.
VALID: Set14 x4 has 14 complete HR/LR pairs.
VALID: BSD100 x4 has 100 complete HR/LR pairs.
VALID: Urban100 x4 has 100 complete HR/LR pairs.

All x3 and x4 datasets passed validation: 438 image evaluations queued.


## Set5 pilot gate

The x2 run never exercised the cascade code, so this checks the new paths before committing ~8 hours of compute. It uses a single untimed run per image and must pass before the full run below.

It confirms: exact HR output size, the correct per-scale strategy label, the correct native pass count (1 for x3, 2 for x4), and a sane PSNR against a bicubic reference on the same images.

In [6]:
pilot_failures = []
for scale in SCALES:
    validation = validations[('Set5', scale)]
    pairs = pair_image_paths(validation.hr_directory, validation.lr_directory)
    nedi_config = NEDIEvaluationConfig(
        dataset='Set5', scale=scale, window_size=WINDOW_SIZE,
        edge_threshold=EDGE_THRESHOLD, warmup_runs=0, timed_runs=1,
    )
    bicubic_config = BicubicEvaluationConfig(
        dataset='Set5', scale=scale, warmup_runs=0, timed_runs=1,
    )
    print(f'\n--- Set5 x{scale} pilot ---')
    for hr_path, lr_path in pairs:
        nedi_record = evaluate_nedi_image(hr_path, lr_path, nedi_config)
        bicubic_record = evaluate_bicubic_image(hr_path, lr_path, bicubic_config)

        checks = {
            'scale label': nedi_record['scale'] == f'x{scale}',
            'strategy label': nedi_record['nedi_scale_strategy'] == NEDI_SCALE_STRATEGIES[scale],
            'native passes': nedi_record['nedi_native_passes'] == NEDI_NATIVE_PASSES[scale],
            'no numerical fallbacks': nedi_record['nedi_numerical_fallback_count'] == 0,
            'plausible psnr': 15.0 < float(nedi_record['psnr_y']) < 45.0,
        }
        failed = [name for name, passed in checks.items() if not passed]
        if failed:
            pilot_failures.append((scale, hr_path.name, failed))

        delta = float(nedi_record['psnr_y']) - float(bicubic_record['psnr_y'])
        print(
            f"  {hr_path.name:<16} HR={nedi_record['hr_width']}x{nedi_record['hr_height']} "
            f"native={nedi_record['nedi_native_width']}x{nedi_record['nedi_native_height']} "
            f"resized={nedi_record['nedi_dimension_adjustment']} "
            f"passes={nedi_record['nedi_native_passes']} "
            f"PSNR-Y={float(nedi_record['psnr_y']):.4f} "
            f"(bicubic {float(bicubic_record['psnr_y']):.4f}, delta {delta:+.4f} dB)"
            + ('  <-- FAILED: ' + ', '.join(failed) if failed else '')
        )

if pilot_failures:
    raise RuntimeError(f'Pilot gate failed; do not start the full run. Failures: {pilot_failures}')
print('\nPILOT PASSED. Dimensions, strategy metadata and scores are correct for x3 and x4.')
print('A NEDI deficit of roughly 0.1-0.5 dB against bicubic is the expected result.')


--- Set5 x3 pilot ---
  baby.png         HR=512x512 native=340x340 resized=True passes=1 PSNR-Y=33.1122 (bicubic 33.9176, delta -0.8054 dB)
  bird.png         HR=288x288 native=192x192 resized=True passes=1 PSNR-Y=31.8001 (bicubic 32.5833, delta -0.7832 dB)
  butterfly.png    HR=256x256 native=170x170 resized=True passes=1 PSNR-Y=23.6379 (bicubic 23.9923, delta -0.3544 dB)
  head.png         HR=280x280 native=186x186 resized=True passes=1 PSNR-Y=32.6459 (bicubic 32.9397, delta -0.2938 dB)
  woman.png        HR=228x344 native=152x228 resized=True passes=1 PSNR-Y=28.2365 (bicubic 28.5442, delta -0.3078 dB)

--- Set5 x4 pilot ---
  baby.png         HR=512x512 native=512x512 resized=False passes=2 PSNR-Y=30.8320 (bicubic 31.7826, delta -0.9506 dB)
  bird.png         HR=288x288 native=288x288 resized=False passes=2 PSNR-Y=29.1983 (bicubic 30.1835, delta -0.9852 dB)
  butterfly.png    HR=256x256 native=256x256 resized=False passes=2 PSNR-Y=21.5582 (bicubic 22.1007, delta -0.5425 dB)
  head.

## Full run

Both scales, all four datasets, both methods. One CSV checkpoint per dataset-scale-method group, rewritten after every image, so an interrupted run resumes by setting `RESUME_RUN_ID` above to this run's folder name.

In [7]:
def load_checkpoint(path):
    if not path.exists():
        return []
    with path.open(newline='', encoding='utf-8') as file:
        return list(csv.DictReader(file))

def run_dataset_method(dataset, scale, method):
    validation = validations[(dataset, scale)]
    pairs = pair_image_paths(validation.hr_directory, validation.lr_directory)
    checkpoint_csv = METRICS_ROOT / f'{dataset}_x{scale}_{method}_aws_final.csv'
    records = load_checkpoint(checkpoint_csv)
    completed_images = {record['image'] for record in records}
    pending_pairs = [(hr, lr) for hr, lr in pairs if hr.name not in completed_images]

    if method == 'bicubic':
        config = BicubicEvaluationConfig(
            dataset=dataset, scale=scale,
            warmup_runs=WARMUP_RUNS, timed_runs=TIMED_RUNS,
        )
        evaluator = evaluate_bicubic_image
    else:
        config = NEDIEvaluationConfig(
            dataset=dataset, scale=scale, window_size=WINDOW_SIZE,
            edge_threshold=EDGE_THRESHOLD,
            warmup_runs=WARMUP_RUNS, timed_runs=TIMED_RUNS,
        )
        evaluator = evaluate_nedi_image

    print(f'{dataset} x{scale} {method}: {len(completed_images)}/{len(pairs)} already complete.')
    with ProcessPoolExecutor(max_workers=WORKERS) as executor:
        futures = {executor.submit(evaluator, hr, lr, config): hr.name for hr, lr in pending_pairs}
        for future in as_completed(futures):
            record = future.result()
            record['compute_instance'] = COMPUTE_INSTANCE
            record['execution_mode'] = 'parallel_process_workers'
            record['worker_count'] = WORKERS
            records.append(record)
            records.sort(key=lambda item: item['image'])
            write_results_csv(records, checkpoint_csv, overwrite=True)
            print(
                f'{dataset} x{scale} {method}: {len(records)}/{len(pairs)} — {record["image"]} — '
                f'PSNR-Y={float(record["psnr_y"]):.4f}, '
                f'time={float(record["latency_mean_ms"]) / 1000:.2f}s'
            )
    return records

records_by_scale = {scale: {'bicubic': [], 'nedi': []} for scale in SCALES}
for scale in SCALES:
    for dataset in DATASETS:
        records_by_scale[scale]['bicubic'].extend(run_dataset_method(dataset, scale, 'bicubic'))
        records_by_scale[scale]['nedi'].extend(run_dataset_method(dataset, scale, 'nedi'))

for scale in SCALES:
    print(f"Completed {len(records_by_scale[scale]['nedi'])} NEDI and "
          f"{len(records_by_scale[scale]['bicubic'])} bicubic x{scale} evaluations.")

Set5 x3 bicubic: 0/5 already complete.
Set5 x3 bicubic: 1/5 — butterfly.png — PSNR-Y=23.9923, time=0.00s
Set5 x3 bicubic: 2/5 — head.png — PSNR-Y=32.9397, time=0.00s
Set5 x3 bicubic: 3/5 — bird.png — PSNR-Y=32.5833, time=0.00s
Set5 x3 bicubic: 4/5 — woman.png — PSNR-Y=28.5442, time=0.00s
Set5 x3 bicubic: 5/5 — baby.png — PSNR-Y=33.9176, time=0.00s
Set5 x3 nedi: 0/5 already complete.
Set5 x3 nedi: 1/5 — butterfly.png — PSNR-Y=23.6379, time=1.42s
Set5 x3 nedi: 2/5 — head.png — PSNR-Y=32.6459, time=1.60s
Set5 x3 nedi: 3/5 — bird.png — PSNR-Y=31.8001, time=1.81s
Set5 x3 nedi: 4/5 — woman.png — PSNR-Y=28.2365, time=1.58s
Set5 x3 nedi: 5/5 — baby.png — PSNR-Y=33.1122, time=5.26s
Set14 x3 bicubic: 0/14 already complete.
Set14 x3 bicubic: 1/14 — coastguard.png — PSNR-Y=26.5051, time=0.00s
Set14 x3 bicubic: 2/14 — comic.png — PSNR-Y=23.0799, time=0.00s
Set14 x3 bicubic: 3/14 — baboon.png — PSNR-Y=23.2053, time=0.00s
Set14 x3 bicubic: 4/14 — face.png — PSNR-Y=32.8433, time=0.00s
Set14 x3 bicubic

## Summaries

Separate per-scale NEDI summaries, a combined x3+x4 NEDI summary, and per-scale AWS bicubic/NEDI comparison summaries for the timing tables.

In [8]:
all_nedi_records = []
all_bicubic_records = []

for scale in SCALES:
    nedi_records = records_by_scale[scale]['nedi']
    bicubic_records = records_by_scale[scale]['bicubic']
    all_nedi_records.extend(nedi_records)
    all_bicubic_records.extend(bicubic_records)

    write_results_csv(nedi_records, METRICS_ROOT / f'nedi_x{scale}_all_images_final.csv', overwrite=True)
    write_results_csv(bicubic_records, METRICS_ROOT / f'bicubic_x{scale}_aws_all_images_final.csv', overwrite=True)
    write_results_csv(summarize_results(nedi_records), METRICS_ROOT / f'nedi_x{scale}_summary_final.csv', overwrite=True)
    write_results_csv(
        summarize_results(bicubic_records + nedi_records),
        METRICS_ROOT / f'x{scale}_aws_comparison_summary.csv',
        overwrite=True,
    )

dataset_order = {name: index for index, name in enumerate(DATASETS)}
combined_summary = summarize_results(all_nedi_records)
combined_summary.sort(key=lambda row: (int(row['scale'][1:]), dataset_order[row['dataset']]))
write_results_csv(all_nedi_records, METRICS_ROOT / 'nedi_x3_x4_all_images_final.csv', overwrite=True)
write_results_csv(combined_summary, METRICS_ROOT / 'nedi_x3_x4_summary_final.csv', overwrite=True)

bicubic_lookup = {
    (row['dataset'], row['scale']): row
    for row in summarize_results(all_bicubic_records)
}
for row in combined_summary:
    baseline = bicubic_lookup[(row['dataset'], row['scale'])]
    print(
        f"{row['dataset']} {row['scale']}: images={row['image_count']}, "
        f"PSNR-Y={row['psnr_y']:.4f}, SSIM-Y={row['ssim_y']:.4f}, "
        f"PSNR-RGB={row['psnr_rgb']:.4f}, SSIM-RGB={row['ssim_rgb']:.4f}, "
        f"latency={row['latency_mean_ms']:.2f} ms  "
        f"[bicubic PSNR-Y={baseline['psnr_y']:.4f}, delta={row['psnr_y'] - baseline['psnr_y']:+.4f} dB]"
    )

print(f'\nAll metrics written under: {METRICS_ROOT}')
print('Copy these into the repository results/ folder when the run finishes.')

Set5 x3: images=5, PSNR-Y=29.8865, SSIM-Y=0.8596, PSNR-RGB=28.0811, SSIM-RGB=0.8234, latency=2334.02 ms  [bicubic PSNR-Y=30.3954, delta=-0.5089 dB]
Set14 x3: images=14, PSNR-Y=27.2957, SSIM-Y=0.7588, PSNR-RGB=25.5962, SSIM-RGB=0.7205, latency=5026.18 ms  [bicubic PSNR-Y=27.6226, delta=-0.3269 dB]
BSD100 x3: images=100, PSNR-Y=26.8506, SSIM-Y=0.7174, PSNR-RGB=25.4893, SSIM-RGB=0.6912, latency=3381.12 ms  [bicubic PSNR-Y=27.2004, delta=-0.3498 dB]
Urban100 x3: images=100, PSNR-Y=24.3816, SSIM-Y=0.7286, PSNR-RGB=22.9291, SSIM-RGB=0.7048, latency=17026.02 ms  [bicubic PSNR-Y=24.4482, delta=-0.0666 dB]
Set5 x4: images=5, PSNR-Y=27.7473, SSIM-Y=0.7983, PSNR-RGB=25.9372, SSIM-RGB=0.7461, latency=5963.08 ms  [bicubic PSNR-Y=28.4293, delta=-0.6819 dB]
Set14 x4: images=14, PSNR-Y=25.6970, SSIM-Y=0.6863, PSNR-RGB=24.0211, SSIM-RGB=0.6410, latency=12813.97 ms  [bicubic PSNR-Y=26.0948, delta=-0.3979 dB]
BSD100 x4: images=100, PSNR-Y=25.5321, SSIM-Y=0.6448, PSNR-RGB=24.1211, SSIM-RGB=0.6107, latency